<a href="https://colab.research.google.com/github/drdave-teaching/OPIM3510-BDA1/blob/main/M1_Python_Bootcamp/Week04_Dirty_Data/Project_DirtyEnergyWeather_blank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dirty Data: Weather and Energy
----------------------------------
**OPIM 3510 Python Bootcamp · Dr. Dave Wanik · University of Connecticut** (optional project)

The purpose of this notebook is to give you experience on applying the tools and methods you've learned about to real-world data. We will show how to:
* clean datasets and resample to the same temporal resolution
* make a map and color by a continuous numeric value
* create compelling visualizations and features
* selecting data based on a condition(s)
* joining datasets
* cleaning/processing text columns

Here we go!

# Import Modules
You will always have these three at the top of your script.

# Download the energy data from ISO-NE

There is wonderful data all around you - here is some energy data from ISO New England.

* https://www.iso-ne.com/isoexpress/web/reports/load-and-demand

<center>

!['ISO-Logo'](https://upload.wikimedia.org/wikipedia/commons/thumb/a/a4/ISO_New_England.svg/1280px-ISO_New_England.svg.png)

**Figure:** the logo for ISO New England

![Service area](https://www.iso-ne.com/static-assets/img/ISO038-new-england-load-map3.png)

**Figure:** the service area for ISO New England

![command center](http://d279m997dpfwgl.cloudfront.net/wp/2017/05/isonewengland_control_room_v1-1000x667.jpg)

**Figure:** a view of the command center!

This website says you have to use the captcha and download the data - but you can hack it!

Look at the path from the website:

`https://www.iso-ne.com/transform/csv/hourlysystemdemand?start=20200825&end=20200829`

See the start and end time? Hack it and get it to match your dataset for any time period.

In [ ]:
# i copy/pasted the link then downloaded/dragged onto my google drive
# made a shareable link
# https://drive.google.com/file/d/1caaX_3WqgU3WBBohvIDpIfB4_IhATluB/view?usp=sharing


In [ ]:
# you can always try to open in excel, otherwise skips rows one at a time
# or try to read using read_table() and figure out what to skip and where the header is


In [ ]:
# let's explore the energy data


This dataset looks nice and clean. We have one column for the date and one column for the hour. We will need to bring these together.

* https://stackoverflow.com/questions/17978092/combine-date-and-time-columns-using-python-pandas

In [ ]:
# let's get the hours on the scale of 0 to 23

# now subtract one

# check your work


In [ ]:
# convert to a date column


Make a new datetime column
* https://stackoverflow.com/questions/52281633/pandas-convert-the-number-of-the-day-and-the-hour

In [ ]:
# now push them together!


It's often useful to make the datetime column the index. You can also make new columns for month, day, year and more if you wanted to!

In [ ]:

# check your work!


Now you can make a nice plot of the energy demand. 

Always be conscious of what time zone your data comes from - we will skip this for now, but in the real world, you may need to tinker with this.

In [ ]:
# here is a plot of energy demand!


Later on, we can join this with temperature data and explore it.

On your own - can you make this plot with better axis titles/labels?

# Download weather data to the local runtime

Feeling slick? Make a new variable for the dates so that you don't need to tinker/dig through your script to update the dates.

In [ ]:
"""
Example script that scrapes data from the IEM ASOS download service
"""
from __future__ import print_function
import json
import time
import datetime

# Python 2 and 3: alternative 4
try:
    from urllib.request import urlopen
except ImportError:
    from urllib2 import urlopen

# Number of attempts to download data
MAX_ATTEMPTS = 6
# HTTPS here can be problematic for installs that don't have Lets Encrypt CA
SERVICE = "http://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"


def download_data(uri):
    """Fetch the data from the IEM
    The IEM download service has some protections in place to keep the number
    of inbound requests in check.  This function implements an exponential
    backoff to keep individual downloads from erroring.
    Args:
      uri (string): URL to fetch
    Returns:
      string data
    """
    attempt = 0
    while attempt < MAX_ATTEMPTS:
        try:
            data = urlopen(uri, timeout=300).read().decode("utf-8")
            if data is not None and not data.startswith("ERROR"):
                return data
        except Exception as exp:
            print("download_data(%s) failed with %s" % (uri, exp))
            time.sleep(5)
        attempt += 1

    print("Exhausted attempts to download, returning empty data")
    return ""


def get_stations_from_filelist(filename):
    """Build a listing of stations from a simple file listing the stations.
    The file should simply have one station per line.
    """
    stations = []
    for line in open(filename):
        stations.append(line.strip())
    return stations


def get_stations_from_networks():
    """Build a station list by using a bunch of IEM networks."""
    stations = []
    # states = """AK AL AR AZ CA CO CT DE FL GA HI IA ID IL IN KS KY LA MA MD ME
    #  MI MN MO MS MT NC ND NE NH NJ NM NV NY OH OK OR PA RI SC SD TN TX UT VA VT
    #  WA WI WV WY"""
    states = 'CT'
    # IEM quirk to have Iowa AWOS sites in its own labeled network
    networks = ["AWOS"]
    for state in states.split():
        networks.append("%s_ASOS" % (state,))

    for network in networks:
        # Get metadata
        uri = (
            "https://mesonet.agron.iastate.edu/geojson/network/%s.geojson"
        ) % (network,)
        data = urlopen(uri)
        jdict = json.load(data)
        for site in jdict["features"]:
            stations.append(site["properties"]["sid"])
    return stations


def download_alldata():
    """An alternative method that fetches all available data.
    Service supports up to 24 hours worth of data at a time."""
    # timestamps in UTC to request data for
    startts = datetime.datetime(2020, 8, 25)
    endts = datetime.datetime(2020, 8, 29)
    interval = datetime.timedelta(hours=24)

    service = SERVICE + "data=all&tz=Etc/UTC&format=comma&latlon=yes&"

    now = startts
    while now < endts:
        thisurl = service
        thisurl += now.strftime("year1=%Y&month1=%m&day1=%d&")
        thisurl += (now + interval).strftime("year2=%Y&month2=%m&day2=%d&")
        print("Downloading: %s" % (now,))
        data = download_data(thisurl)
        outfn = "%s.txt" % (now.strftime("%Y%m%d"),)
        with open(outfn, "w") as fh:
            fh.write(data)
        now += interval


def main():
    """Our main method"""
    # timestamps in UTC to request data for
    startts = datetime.datetime(2020, 8, 25)
    endts = datetime.datetime(2020, 8, 29)

    service = SERVICE + "data=all&tz=Etc/UTC&format=comma&latlon=yes&"

    service += startts.strftime("year1=%Y&month1=%m&day1=%d&")
    service += endts.strftime("year2=%Y&month2=%m&day2=%d&")

    # Two examples of how to specify a list of stations
    stations = get_stations_from_networks()
    # stations = get_stations_from_filelist("mystations.txt")
    for station in stations:
        uri = "%s&station=%s" % (service, station)
        print("Downloading: %s" % (station,))
        data = download_data(uri)
        outfn = "%s_%s_%s.txt" % (
            station,
            startts.strftime("%Y%m%d%H%M"),
            endts.strftime("%Y%m%d%H%M"),
        )
        out = open(outfn, "w")
        out.write(data)
        out.close()


if __name__ == "__main__":
    download_alldata()
    # main()

Look left! All of the data you need is there. 

# Open the data!

Yuck! Looks like it has some nasty junk at the top. We can try to skip lines and read again.

In [ ]:
# you will need to find arguments in read_table() that can help you

## Reading/appending multiple files
Working with each of the individual files can be OK sometimes - other times it's good to merge and read all three files. 

Let's open and append the three files.

See if you can hack this example for TEXT FILES (not .csv). Do you remember how to copy path from a file?

In [ ]:
# glob

Now let's check our work and inspect the data types (dtypes).

Wow! That is a lot of data. 

## value_counts, nunique and unique
Let's see how many airports there are.

Looks like some stations have a lot of data, while others have just ONE observation. We can clean this up and select those that have more than X observations. A histogram and summary statistics can help here. 

## summary statistics

On average, it looks like the average station has around 100 observations - this makes sense for three days of data.

Sometimes it's nice to view these data as a histogram - different patterns can emerge. Play with the `bins` argument to get some more flexibility.

Aha! We see quite a few outliers here... some stations have HUNDREDS of records while most seem to be between 0 and 250. A boxplot could be a nice way of summarizing the middle 50% of the data. 

## Boxplot


If you wanted to, you could customize the Y axis so that it was easier to read (yes, there are outliers - but we might be more interested in the IQR and whiskers.

BTW - why don't you check the documentation and see what the whiskers mean... are they +/- 1.5*IQR or something different?

## kde
Which also look A LOT like kernel density plots!

# Correlation
It's interesting to look at the correlation among variables.

Whenever you have latitiude and longitude in your dataset, try to explore it! Here are all of the points - should form a map of the World.

AMAZING!!! Later on when we clean up our data types, we will be able to look at temperatures all around the World (or all across New England, whatever we want to do.)

# Data Types (using `df.info()`) and the hunt for missing values

As you may now, missing data can be identified with 'M' or '999' or 'NA' or 'NaN' or a whole mess of other garbage. When a dataset is this big, you need to be clever with how you identify missing values.

If missing values are common, you also might be able to investigate with `value_counts`

Now, you have a choice when there is a constant in the dataset - you get replace all 'M' with NaN or 0 (dangerous! there are better ways.) I find the best way is to convert the numeric columns all at once and COERCE the errors.

What does COERCE do? It tries to force the values to take a numeric type. This can save a lot of replacement text.

## replacing the 'M' in the 'tmpf' column

For completeness, let's show how to replace some missing values one at a time in just one column.

When searching for an answer, you might find related questions that can help you solve the problem.
* https://stackoverflow.com/questions/38516481/trying-to-remove-commas-and-dollars-signs-with-pandas-in-python/46430853

Now you can coerce this column to a numeric data type.

LEt's count the missing values in this column now that things have been coerced.

You could also find this using df.info()...

# Data Dictionary
When possible, it's always good to look at the data dictionary - let's find out which columns should be numeric and convert them.
* https://mesonet.agron.iastate.edu/request/download.phtml

Which columns are numeric and which are objects/text?

## Converting multiple columns to numeric

StackOverflow is filled with solid gold - your question has likely already been solved :)

* https://stackoverflow.com/questions/36814100/pandas-to-numeric-for-multiple-columns

Hack it and go! See the power of 'apply' for repeating certain operations? No need for a for loop. Always strive to avoid using a for loop whenever possible.

# [optional] Make a map
Now that you have wonderful data - go do something with it! You already saw that your data had a geospatial component - why not try making a map of max and min temperature across time?

We can try many different approaches, but to start, why not just make a scatterplot like we did before and color it by some z value.

Now make a scatterplot of lat vs. lon and color by tmpf (max temperature!) See how your data comes to life?
* https://seaborn.pydata.org/generated/seaborn.scatterplot.html

**On your own:** Why not redo the code and see where the coldest temperatures are? Add labels and a title.

Try to zoom in on Connecticut. 

Here's how you can make a basemap with it.

Try to zoom in on Connecticut - Stamford's lat/lon is 41.0534° N, 73.5387° W.

Looks a little crude, but you can tinker on your own!

# Subset one or more airports from Connecticut
For ease, let's just subset a few airports from the Connecticut area. Bradley (BDL) near Hartford, Tweed (HVN) near New Haven, and White Plains (HPN) next to Stamford are great options for us.

In [ ]:
# you'll notice I always overwrite the variable 'tmp' 
# because I'm lazy - just be careful not to overwrite something you need
# if you do, just restart and run your script or choose a different name

Here are a few ways you can select data - each has a pro or con. `.isin()` is a great option when you have a lot of unique values that you need to subset, otherwise you can subset with a logical.

Here is how to subset based on a logical with multiple conditions. Note how each condition is wrapped in `()` and we use the OR operator `|`.

In [ ]:
# see how I use lower and upper case letters for my variables?
# this is just my style - you could also use underscores

# select one station


When it's just one station you can do a lot right from the dataframe. A lot of `matplotlib` functionality can be controlled right from the pandas dataframe itself. Less code is better and pandas is AWESOME!

This plot would probably be a lot more interesting if we had a date on the X axis, which will require us to convert to a datetime object.

As you know, it's sometimes useful to set the index as the datetime column, because it allows you to take full advantage of the pandas time series functionality.
* https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html

Now your code knows what to do - make a plot of temperature!

In [ ]:
# notice how the plot changed with the proper axis! 
# this plot is way different than the one above
# this is due to clumpy/frequent observations


## Selecting multiple stations

In [ ]:
# select three stations


It would be a fruitful exercise to go from long to wide now! When you do, you may come up with a multi-index. You can read about this on your own. For now we will collapse the index so we have one set of column names.

* https://stackoverflow.com/questions/34988040/long-to-wide-data-pandas

In [ ]:
# first, make sure that 'valid' is a datetime for easy analysis/resampling later on


In [ ]:
# let's make a pivot table

# check your work


The concept of a multi-index can be annoying... we can squish it down further if we want and rename the columns.

In [ ]:
#reset multicolumns
# this strips the whitespace and joins them together with a _



Now you can resample and explore!

See how easy it is to line up variables when they are on the same time scale! Looks like HVN was a little bit warmer than BDL. You can also customize the plot on your own.

# Join weather and energy data together
The energy data is VERY CLEAN. It is on an hourly resoultion. 

The weather data is more chaotic - you will need to resample to hourly resolution. 

Fortunately, pandas has a whole mess of tools that are useful for resampling time series data.

* https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html

In [ ]:
# we can make a copy of the dataset for clearer names


Now everything is at hourly resolution - we are ready for a join!
* https://www.statology.org/pandas-merge-on-index/

In [ ]:
# we will do the join on the index!
# if you are doing it by a column, this will be different syntax


# Time to Analyze!
Now there are infinite ways that one can tease out the relationship between energy and weather. 

You should anticipate how many rows and columns there should be - and get ready to clean up even further to make your life easier.

On your own: Add a linear trend line and the equation of the line to this plot (like Excel.) You can also try to add a polynomial. 

* https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html
* https://stackoverflow.com/questions/49460596/add-trendline-with-equation-in-2d-array

We can save a copy of the dataset and keep working on it later.

In [ ]:
 # this will download to the runtime on the left

Let's think about all the things we can do with this dataset... you likely don't know how to do all of these, so search for the answer when you are stuck! 

1. Day of the week (Monday, Tuesday, Wednesday) and color code the scatterplot of tmpf vs. MWh by day of week
2. How many hours were above 70 degF in your dataset?
3. Calculate the mean or max consecutive values above a threshold (like 70 degF vs. 80 degF)
4. Shift the weather or energy variables by 1, 6 or 12 time steps and make scatterplots.
5. What is the correlation between previous one hour and current hour? Six hours and current hour? 12 hours and current hour? 24 hours and current hour? Which of these do you think would be a fair baseline model?
6. Calculate a 2, 3, 6, 12 hr rolling means as new columns in your df for both weather and energy - what do the missing values look like in your df after making these features (and think about WHY you have missing values)
7. What is the greatest hourly change? 6 hourly change? You can think of this as a slope or difference between two rows.
8. What is the greatest hourly percentage change for temperature or energy? Create a histogram of this variable.
9. Identify all of the local minima and maxima in the temperature and energy data. Plot them as points on the time series plot.
10. Return the time that temperature climbs above 70 degF each day (if at all).




In [ ]:
# anyone else have something they want to try?

# Text Analytics: Working with METAR

The METAR data is quite interesting! It's the way pilots talk to teach other about the weather. Let's look at one.

* https://en.wikipedia.org/wiki/METAR#METAR_conventions

In [ ]:
# first METAR


Let's count the number of words in each cell.

In [ ]:
# here is a nice table (just printed out, not saved anywhere)


In [ ]:
# at first look it looks like 12 in each one... consistent!


Looks there are some that are 12, but a few that are quite long (even if rare!) 

WHOA! So there are some random ones that are quite long... why don't we save the length as a field then subset the long records to see what they look like? 

In [ ]:
# make a new column with the length of the METAR


In [ ]:
# subset to examine


Since there aren't that many, maybe we can look at the METARs for trends. 

In [ ]:
# it can be tough for pandas to print the entire series


Despite our best efforts, it probably would be tough to line up all of these columns as new variables - we would hope for the 12 standard variables listed on the Wikipedia page. So, for METARs where length is equal to 12, let's parse and make 12 new variables which we can clean up later!

In [ ]:
# it's not pretty, but let's delete rows that have more than 12 metars - too complicated for us right now


In [ ]:
# we lost a few records - this is OK for now - just a demonstration


Now let's create 12 new variables in `tmp` that we hope line up with the METAR data.
* https://stackoverflow.com/questions/14745022/how-to-split-a-dataframe-string-column-into-two-columns

In [ ]:
# i hacked the code to split by a delimeter


Now let's join it back to the original dataset...

In [ ]:
# make the new columns (for now, with boring names)


Can you read Wikipedia and figure out how to decipher the METAR into your other weather variables? This is a great exercise!

On your own, you can make all text upper case or lower case, or select values from METAR based on a condition.